In [0]:
from TornAPI.Torn import Faction
from pyspark.sql.functions import lit, explode, from_unixtime, to_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, BooleanType, ArrayType

import datetime as dt

faction_api = Faction(dbutils.secrets.get("Personal", "TornAPI"))

In [0]:
if spark.catalog.tableExists("torn.faction.attacks"):
    current_data = spark.read.table("torn.faction.attacks")
    max_date = current_data.select("started").agg({"started":"max"}).collect()
    current_max_date = max_date[0]["max(started)"]
    id_list = [id_num[0] for id_num in current_data.select("id").collect()]
else:
    current_max_date = 1681484237
    id_list = []

In [0]:

while current_max_date <= (dt.datetime.today() + dt.timedelta(days=-1)).timestamp():

    attack_data = faction_api.get_attacks(ts_from=current_max_date ,sort="ASC")

    schema = StructType([
    StructField("id", IntegerType()),
    StructField("code", StringType()),
    StructField("started", IntegerType()),
    StructField("ended", IntegerType()),
    StructField("result", StringType()),
    StructField("attacker", StructType([
                StructField("id", IntegerType()),
                StructField("name", StringType()),
                StructField("level", IntegerType()),
                StructField("faction", StructType(
                    [
                        StructField("id", IntegerType()),
                        StructField("name", StringType())
                    ]))])
                ),
    StructField("defender", StructType([
            StructField("id", IntegerType()),
            StructField("name", StringType()),
            StructField("level", IntegerType()),
            StructField("faction", StructType(
                    [
                        StructField("id", IntegerType()),
                        StructField("name", StringType())
                    ])
                )
        ]
    )),

    StructField("respect_gain", FloatType()),
    StructField("respect_loss", FloatType()),
    StructField("chain", IntegerType()),
    StructField("is_interrupted", BooleanType()),
    StructField("is_stealthed", BooleanType()),
    StructField("is_raid", BooleanType()),
    StructField("is_ranked_war", BooleanType()),
    StructField("modifiers", StructType([
        StructField("fair_fight", FloatType()),
        StructField("war", FloatType()),
        StructField("retaliation", FloatType()),
        StructField("group", FloatType()),
        StructField("overseas", FloatType()),
        StructField("chain", FloatType()),
        StructField("warlord", FloatType())]
    ))  
    ])

    sp_attacks = spark.createDataFrame(attack_data["attacks"], schema= schema)

    sp_attacks = sp_attacks.filter(~sp_attacks.id.isin(id_list))

    sp_attacks.write.format("delta").mode("append").saveAsTable("torn.faction.attacks")

    current_data = spark.read.table("torn.faction.attacks")
    max_date = current_data.select("started").agg({"started":"max"}).collect()
    current_max_date = max_date[0]["max(started)"]
    id_list = [id_num[0] for id_num in current_data.select("id").collect()]


In [0]:
sp_df = spark.read.table("torn.faction.attacks")
sp_df = sp_df.dropDuplicates()

sp_df.write.format("delta").mode("overwrite").saveAsTable("torn.faction.attacks")
